## Environment setup

Run the next cell once per fresh kernel to install notebook dependencies.
If you install packages in the active kernel, restart the kernel and rerun all cells.

In [ ]:
# Cloud and local notebooks: install required runtime dependencies.
!pip install -q protein-quest[nb]

# AlphaFold

You can download and filter AlphaFold files on confidence.

In [2]:
# Generic imports
import logging
from pathlib import Path
from pprint import pprint

logging.basicConfig(level=logging.WARNING)
# Set to WARNING to see only warnings
# Set to INFO to see sparql queries
# Set to DEBUG to see raw results


## Download Alphafold files

In [3]:
from protein_quest.alphafold.fetch import fetch_many_async

In [4]:
save_dir = Path("alphafold_files")

To download the summary, the cif and predicted Aligned error document (peaDoc) file for 3 AlphaFold entries given their uniprot accessions.


In [7]:
summaries = [
    s async for s in fetch_many_async(["A1YPR0", "O60481", "P50613"], save_dir, formats={"summary", "cif", "paeDoc"})
]
pprint(summaries)

Fetching Alphafold summaries: 100%|██████████| 3/3 [00:00<00:00, 12.32it/s]

[AlphaFoldEntry(uniprot_accession='A1YPR0',
                summary=EntrySummary(allVersions=[1, 2, 3, 4, 5, 6],
                                     bcifUrl=URL('https://alphafold.ebi.ac.uk/files/AF-A1YPR0-F1-model_v6.bcif'),
                                     cifUrl=URL('https://alphafold.ebi.ac.uk/files/AF-A1YPR0-F1-model_v6.cif'),
                                     entityType='protein',
                                     fractionPlddtConfident=0.26,
                                     fractionPlddtLow=0.099,
                                     fractionPlddtVeryHigh=0.089,
                                     fractionPlddtVeryLow=0.553,
                                     globalMetricValue=56.03,
                                     isUniProt=True,
                                     latestVersion=6,
                                     modelCreatedDate='2025-08-01T00:00:00Z',
                                     modelEntityId='AF-A1YPR0-F1',
                              

In [8]:
!ls -sh {save_dir}

total 3.3M
4.0K A1YPR0.json
556K AF-A1YPR0-F1-model_v6.cif
1.1M AF-A1YPR0-F1-predicted_aligned_error_v6.json
412K AF-O60481-F1-model_v6.cif
628K AF-O60481-F1-predicted_aligned_error_v6.json
324K AF-P50613-F1-model_v6.cif
276K AF-P50613-F1-predicted_aligned_error_v6.json
8.0K O60481.json
4.0K P50613.json


## Filter AlphFold structure files on confidence

Filter AlphaFold mmcif/PDB files by confidence (plDDT). Passed files are written with residues below threshold removed.

In [9]:
from protein_quest.alphafold.confidence import ConfidenceFilterQuery, filter_files_on_confidence

Take one of the downloaded files

In [10]:
input_files = [entry.cif_file for entry in summaries if entry.cif_file is not None]
input_files

[PosixPath('alphafold_files/AF-A1YPR0-F1-model_v6.cif'),
 PosixPath('alphafold_files/AF-O60481-F1-model_v6.cif'),
 PosixPath('alphafold_files/AF-P50613-F1-model_v6.cif')]

We only write a filtered cif file when in the input file there are between 100 and 1000 residues that have a pLDDT score above 50.

In [11]:
query = ConfidenceFilterQuery(confidence=80, min_residues=100, max_residues=1000)

In [12]:
output_dir = Path("./filtered")
output_dir.mkdir(exist_ok=True)
result = filter_files_on_confidence(input_files, query, output_dir)

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:34887
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:41671'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:38889'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:37481'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:36369'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45561'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:37863'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39747'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:44891'
INFO:distributed.nanny:        Start Na

In [13]:
list(
    filter_files_on_confidence(
        input_files, ConfidenceFilterQuery(confidence=80, min_residues=100, max_residues=1000), output_dir
    )
)

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:46063
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:38027'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:44111 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:44111
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:48678
INFO:distributed.scheduler:Receive client connection: Client-cd3d96b6-2210-11f1-9b29-00155d9940c2
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:48682
INFO:distributed.scheduler:Registering Worker plugin forward-logging-<root>
INFO:distributed.scheduler:Remove client Client-cd3d96b6-2210-11f1-9b29-00155d9940c2
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:48682; closing.
INFO:distributed.scheduler:

[ConfidenceFilterResult(input_file='AF-A1YPR0-F1-model_v6.cif', count=199, filtered_file=PosixPath('filtered/AF-A1YPR0-F1-model_v6.cif')),
 ConfidenceFilterResult(input_file='AF-O60481-F1-model_v6.cif', count=67, filtered_file=None),
 ConfidenceFilterResult(input_file='AF-P50613-F1-model_v6.cif', count=248, filtered_file=PosixPath('filtered/AF-P50613-F1-model_v6.cif'))]

2 files have passed, but 1 file only has 75 high confidence residues so it is discarded.

## Visualize a structure with Mol*

Use [molviewspec](https://molstar.org/mol-view-spec/) to visualize one of the structures.

In [ ]:
from molviewspec import create_builder, molstar_notebook

# TODO pick first cif file from alphafold_files/
structure_file = Path("alphafold_files/AF-A1YPR0-F1-model_v6.cif")

builder = create_builder()
builder.download(url=structure_file.name).parse(format="mmcif").model_structure().component().representation().color(color="blue")
# TODO color by confidence

molstar_notebook(
    state=builder.get_state(), data={structure_file.name: structure_file.read_bytes()}
)

<IPython.core.display.Javascript object>